In [54]:
import pandas as pd
import numpy as np
import os
import pickle
from pathlib import Path

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import precision_score, recall_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight


In [55]:
def load_data(csv_path, images_base):

    articles = pd.read_csv(csv_path)

    # Drop rows where index_group_name is 'Sport'
    value_counts = articles['index_group_name'].value_counts()
    rare_classes = value_counts[value_counts < 10].index.tolist()

    # Add divided to the rare classes
    rare_classes.append('Divided')
    print(f"rare calsses: {rare_classes}")

    articles = articles[~articles['index_group_name'].isin(rare_classes)].reset_index(drop=True)


    def get_image_path(article_id):
        article_str = str(article_id).zfill(10)
        folder = article_str[:3]
        img_file = os.path.join(images_base, folder, f"{article_str}.jpg")
        return img_file if os.path.exists(img_file) else None

    articles['image_path'] = articles['article_id'].apply(get_image_path)
    articles = articles.dropna(subset=['image_path'])

    return articles


In [56]:
def preprocess_images(image_paths, target_size = (256,256)):

    X = np.array([img_to_array(load_img(p, target_size=target_size))/255.0 for p in image_paths])
    return X

In [57]:
def evaluate_model(generator, model = None):
    """
    Evaluate a model on a generator.
    Returns loss, accuracy, precision, recall, confusion matrix.
    """

    if model == None:
        model = load_model('best_model_keras')

    steps = len(generator)
    loss, accuracy = model.evaluate(generator, steps=steps, verbose=1)

    # Get predictions
    y_true = generator.labels
    y_pred_probs = model.predict(generator, steps=steps, verbose=1)
    if y_pred_probs.shape[1] > 1:  # multi-class
        y_pred = np.argmax(y_pred_probs, axis=1)
    else:  # binary
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()

    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')

    print(f"{'='* 100}")
    print(f"acuuracy: {accuracy}")
    print(f"precision: {precision}")
    print(f"recall: {recall}")
    print(f"{'='* 100}")

    #return loss, accuracy, precision, recall


In [58]:
def train_gender_model(
        csv_path,
        images_base,
        model_path="../models/best_gender_classifier.keras",
        encoder_path="../models/gender_label_encoder.pkl",
        target_size=(256,256),
        epochs=40
    ):

    print("📌 Loading data...")
    df = load_data(csv_path, images_base)

    # Label encoding
    le = LabelEncoder()
    df["label"] = le.fit_transform(df["index_group_name"])
    class_names = le.classes_
    num_classes = len(class_names)
    print(f"Detected classes ({num_classes}): {class_names}")

    # Train/val/test split
    train_val_df, test_df = train_test_split(df, test_size=0.1,
                                             stratify=df["label"], random_state=42)
    train_df, val_df = train_test_split(train_val_df, test_size=0.2,
                                        stratify=train_val_df["label"], random_state=42)



    # Compute class weights
    cw = class_weight.compute_class_weight(class_weight='balanced',
                                           classes=np.unique(train_df["label"]),
                                           y=train_df["label"])
    class_weights = dict(enumerate(cw))
    print("Class weights:", class_weights)

    # Generators
    train_datagen = ImageDataGenerator(rescale=1./255)
    val_datagen = ImageDataGenerator(rescale=1./255)
    test_datagen = ImageDataGenerator(rescale=1./255)

    train_generator = train_datagen.flow_from_dataframe(train_df,
                                    x_col="image_path", y_col="index_group_name",
                                    target_size=target_size, class_mode="sparse", batch_size=32, shuffle=True)
    val_generator = val_datagen.flow_from_dataframe(val_df,
                                    x_col="image_path", y_col="index_group_name",
                                    target_size=target_size, class_mode="sparse", batch_size=32, shuffle=False)
    test_generator = test_datagen.flow_from_dataframe(test_df,
                                    x_col="image_path", y_col="index_group_name",
                                    target_size=target_size, class_mode="sparse", batch_size=32, shuffle=False)




    data_augmentation = Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.2),
            layers.RandomZoom(0.2),
            layers.RandomTranslation(0.1, 0.1),

        ])

    print("📌 Building model...")
    base_model = InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=target_size + (3,)
    )
    base_model.trainable = False   # Freeze for first stage
    input_shape = (256,256,3)
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(512, activation='relu')(x)
    #x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    #x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    '''
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(256,256,3)),
        MaxPooling2D(2,2),

        Conv2D(64, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2,2),

        Conv2D(128, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2,2),

        Flatten(),

        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    '''
    #### Compile ####
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint(model_path, save_best_only=True, monitor="val_accuracy", mode="max")
    ]

    print("🚀 Training started...")
    model.fit(train_generator,
              validation_data=val_generator,
              epochs=epochs,
              class_weight=class_weights,
              callbacks=callbacks)

    print("💾 Saving label encoder...")
    with open(encoder_path, "wb") as f:
        pickle.dump(le, f)



    print("🎉 Training complete!")
    return model, test_generator


In [59]:
def classify_gender(image_path,
                    model_path="../models/best_gender_classifier.keras",
                    encoder_path="../models/gender_label_encoder.pkl",
                    target_size=(256,256)):

    # Load model + encoder
    model = load_model(model_path)
    with open(encoder_path, 'rb') as f:
        le = pickle.load(f)

    img = load_img(image_path, target_size=target_size)
    x = img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)

    pred = model.predict(x)
    print(f"pred: {pred}")
    idx = np.argmax(pred, axis=1)[0]

    return le.inverse_transform([idx])[0]

In [65]:
if __name__ == "__main__":
    '''
    # Train model
    model, test_gen = train_gender_model(
        csv_path="../raw_data/articles_filtered.csv",
        images_base="../raw_data/images_filtered",
        model_path="../models/best_gender_classifier.keras",
        encoder_path="../models/gender_label_encoder.pkl",
        epochs=20
    )

    # Evaluate model
    evaluate_model(test_gen, model)
    '''
    # Predict
    test_image = '../raw_data/test_images/shoes_1.jpg'
    gender = classify_gender(str(test_image))
    print(f"gender: {gender}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 828ms/step
pred: [[0.2056091  0.00875631 0.7856345 ]]
gender: Menswear
